## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [6]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
# Use as many cells as you like

Create a pandas DataFrame called nasdaq100_ca containing the nasdaq100_CA.csv file and add a column called "ytd" containing year to date (YTD) performance for each company.

In [7]:
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
price_change = pd.read_csv("nasdaq100_price_change.csv")

In [8]:
nasdaq100_ca = nasdaq100_ca.merge(price_change[['symbol', 'ytd']], on='symbol', how='left')

Use the OpenAI API to classify each stock in the nasdaq_ytd.csv into a sector, saving as a new column in the nasdaq DataFrame called "sector" with the following values: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, or Financial.

In [9]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

sectors = [
    "Technology", "Consumer Cyclical", "Industrials", "Utilities", "Healthcare",
    "Communication", "Energy", "Consumer Defensive", "Real Estate", "Financial"
]

def make_prompt(row):
    return (
        f"Classify the following company into one of these sectors: {', '.join(sectors)}.\n"
        f"Company name: {row['name']}\n"
        f"Description: {row.get('description', '')}\n"
        f"Return only the sector name."
    )

prompts = nasdaq100_ca.apply(make_prompt, axis=1).tolist()

responses = []
for prompt in prompts:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0
    )
    sector = response.choices[0].message.content.strip()
    responses.append(sector)

nasdaq100_ca["sector"] = responses

Use the OpenAI API to provide summary information about the selected Nasdaq companies, recommending the two best sectors and two or more companies per sector, storing as a variable called stock_recommendations.

In [10]:
company_info = ""
for i in range(len(nasdaq100_ca)):
    row = nasdaq100_ca.iloc[i]
    company_info += f"Company: {row['name']}\nSector: {row['sector']}\nDescription: {row.get('description', '')}\n\n"

summary_prompt = (
    "You are a financial analyst. Given the following list of Nasdaq companies with their sectors and descriptions, "
    "provide a concise summary of the companies. Then, recommend the two best sectors for investment based on popularity, performance, or general market trends. "
    "For each of those two sectors, recommend at least three of the most popular or well-known companies from the list. "
    "If there are fewer than three companies in a sector, list all available. "
    "Format your response exactly as follows:\n"
    "Summary: <summary>\n"
    "Best Sectors: <sector 1>, <sector 2>\n"
    "Recommendations:\n"
    "<sector 1>:\n- <company 1>\n- <company 2>\n- <company 3>\n"
    "<sector 2>:\n- <company 4>\n- <company 5>\n- <company 6>\n"
    "For each recommendation, summarize how the company and stock is performing"
    "Only use company names from the provided data. Do not invent companies. Do not include any explanations or extra text outside the specified format.\n"
    "\nHere is the data:\n"
    f"{company_info}"
)

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": summary_prompt}],
    max_tokens=500,
    temperature=0.2
)

stock_recommendations = response.choices[0].message.content.strip()
print(stock_recommendations)

Summary: The list includes companies from various sectors such as Technology, Healthcare, Real Estate, Energy, Consumer Defensive, Communication, Financial, and Consumer Cyclical.

Best Sectors: Technology, Healthcare

Recommendations:
Technology:
- Apple Inc.: Apple Inc. is a well-known technology company that is known for its innovative products and services. The stock has shown strong performance over the years.
- Adobe Inc.: Adobe Inc. is a leading software company in the technology sector. The stock has performed well due to its popular software products.
- Nvidia: Nvidia is a prominent technology company known for its graphics processing units (GPUs) and artificial intelligence technology.

Healthcare:
- Amgen: Amgen is a major player in the healthcare sector, particularly in biotechnology. The company has a strong track record of developing innovative drugs and treatments.
- DexCom: DexCom is a healthcare company specializing in continuous glucose monitoring systems for diabetes